# TILDE (2021)
---
[[paper]](https://arxiv.org/pdf/2104.07223)<br>TILDE = Term Independent Learned DEcoders

TILDE — это инновационный подход к **нейронному разреженному поиску (Neural Sparse Retrieval)**, разработанный исследователями из Carnegie Mellon University. Он использует мощь предварительно обученных языковых моделей для генерации семантически обогащенных разреженных представлений документов и запросов, сочетая эффективность традиционных разреженных методов с семантической глубиной плотных моделей.

### Контекст

В информационном поиске существуют два доминирующих семейства методов:
1.  **Разреженные методы (Sparse Retrieval)**, такие как BM25 и TF-IDF, основаны на лексическом совпадении терминов. Они очень эффективны, легко индексируются (используя инвертированные индексы), интерпретируемы и легко обновляются. Однако их главный недостаток — неспособность обрабатывать семантические несоответствия (например, синонимы, парафразы), что приводит к низкой точности при наличии только концептуального, а не лексического совпадения.
2.  **Плотные методы (Dense Retrieval)**, такие как DPR (2020) и ANCE (2020), используют глубокие нейронные сети (например, BERT) для кодирования запросов и документов в плотные векторы (embeddings). Они отлично справляются с семантическим сопоставлением, но имеют свои недостатки:
    *   Высокие вычислительные затраты на хранение и поиск (использование Approximate Nearest Neighbor (ANN) индексов).
    *   Сложность обновления индекса при добавлении новых документов.
    *   Сниженная интерпретируемость.

Существовала потребность в методе, который мог бы объединить лучшие качества обоих подходов: семантическую мощь плотных моделей с эффективностью, интерпретируемостью и масштабируемостью разреженных систем.

### Идея метода

Основная идея TILDE заключается в использовании предварительно обученного Transformer-энкодера (например, на основе BERT или RoBERTa) не для генерации плотных эмбеддингов всего документа/запроса, а для **предсказания важности каждого термина в словаре** применительно к конкретному документу или запросу. Таким образом, TILDE учится создавать **разреженные векторы терминов**, которые содержат веса для каждого словаря, даже если эти слова не присутствуют в исходном тексте. Это позволяет системе выполнять семантическое расширение запроса/документа, сохраняя при этом формат разреженного представления, пригодный для инвертированных индексов.

### Постановка задачи

Решается задача **ранжирования документов (passage ranking)**: для заданного запроса $Q$ и коллекции документов $D = \{d_1, d_2, ..., d_N\}$, необходимо найти $K$ наиболее релевантных документов.

### Альтернативные методы на момент появления TILDE (2021)

*   **BM25 (1994):** Стандартный алгоритм разреженного поиска, основанный на частоте терминов и длине документов. Высокая эффективность, но низкая семантическая чувствительность.
*   **DPR (2020):** Первая широко принятая модель плотного поиска, использующая двухбашенную архитектуру на основе BERT для создания плотных эмбеддингов запросов и документов, и затем вычисляющая их скалярное произведение. Значительное улучшение семантической релевантности, но с присущими плотным методам недостатками.
*   **ANCE (2020):** Улучшенная версия DPR, использующая асинхронное контрастивное обучение и кэширование негативных примеров для более стабильного и эффективного обучения плотных эмбеддингов.
*   **ColBERT (2020):** Еще один плотный метод, который вместо одного плотного вектора для всего документа генерирует несколько векторов для каждого токена и использует "late interaction" для вычисления схожести, что позволяет более гранулярно оценивать релевантность.

TILDE отличается от этих методов тем, что вместо генерации плотных векторов или полагания на лексическое совпадение, он **обучается генерировать *взвешенные разреженные векторы* из семантически богатой языковой модели**.

### Архитектура модели

Архитектура TILDE состоит из двух основных компонентов:
1.  **Базовый Transformer-энкодер:** Стандартная предварительно обученная Transformer-модель (например, ELECTRA-Large), которая принимает на вход последовательность токенов (запрос или документ) и генерирует контекстуализированные эмбеддинги для каждого токена.
2.  **Проекционный слой (Prediction Head):** Линейный слой, который применяется к выходным эмбеддингам каждого токена от Transformer-энкодера. Этот слой проецирует эмбеддинг токена в вектор размером с размер словаря (vocabulary size). Каждое значение в этом выходном векторе представляет собой **скор (вес)**, ассоциированный с соответствующим термином из словаря.
    *   Пример: если у нас есть документ "The quick brown fox", Transformer-энкодер выдаст эмбеддинги для "The", "quick", "brown", "fox". Каждый из этих эмбеддингов затем проходит через Проекционный слой, выдавая четыре вектора, каждый из которых по размеру равен словарю.
    *   **Агрегация:** Для получения окончательного разреженного представления всего документа/запроса, эти векторы скоров для каждого токена агрегируются. Авторы используют **максимум (max-pooling)** по позициям токенов: для каждого термина из словаря, берется максимальный скор, предсказанный для него любым токеном входной последовательности. Это позволяет выделить наиболее важные "активации" для каждого термина. Полученный вектор и есть разреженное представление.
    *   **Sparsity:** Разреженность достигается за счет использования функции активации (например, ReLU) и/или применения порогов, которые отсекают низкие веса, превращая их в нули.

### Алгоритм обучения

Обучение TILDE происходит с использованием **дистилляции знаний (knowledge distillation)** от мощной **Cross-Encoder модели** (учителя), которая выступает в роли "золотого стандарта" релевантности. Cross-Encoder'ы (например, BERT-Large ре-ранкеры), оценивая пару (запрос, документ) одновременно, достигают наивысшей точности в ранжировании, но слишком медленны для поиска по большому корпусу.

Шаги обучения:
1.  **Формирование обучающих данных:** Для каждого запроса $Q$, берется набор документов $D_{sampled} = \{d_1, ..., d_M\}$, которые включают как релевантные, так и нерелевантные документы.
2.  **Оценка учителем:** Cross-Encoder (учитель) вычисляет **"истинные" релевантности** $S_{teacher}(Q, d_i)$ для каждой пары $(Q, d_i)$ в $D_{sampled}$. Эти скоры нормализуются до вероятностей (например, с помощью softmax), формируя распределение релевантности.
3.  **Генерация разреженных весов TILDE:** Для каждого документа $d_i$ и запроса $Q$ модель TILDE генерирует их разреженные векторные представления: $v(d_i)$ и $v(Q)$.
4.  **Вычисление скора TILDE:** Скор релевантности для пары $(Q, d_i)$ вычисляется как **скалярное произведение (dot product)** разреженных векторов: $S_{TILDE}(Q, d_i) = v(Q) \cdot v(d_i)$. Эти скоры также нормализуются до вероятностей.
5.  **Функция потерь:** Модель TILDE (ученик) обучается минимизировать **KL-дивергенцию (Kullback-Leibler Divergence)** между распределением релевантности, предсказанным учителем ($P_{teacher}(d_i|Q)$), и распределением релевантности, предсказанным TILDE ($P_{TILDE}(d_i|Q)$). Это заставляет TILDE имитировать ранжирующее поведение мощного учителя.
    $$ L = -\sum_{d_i \in D_{sampled}} P_{teacher}(d_i|Q) \log P_{TILDE}(d_i|Q) $$
    Где $P_{teacher}(d_i|Q) = \frac{\exp(S_{teacher}(Q, d_i))}{\sum_{d_j \in D_{sampled}} \exp(S_{teacher}(Q, d_j))}$ и аналогично для $P_{TILDE}$.

### Алгоритм инференса

1.  **Создание индекса документов:**
    *   Для каждого документа $d_i$ в корпусе:
        *   Пропустить $d_i$ через модель TILDE, чтобы получить его разреженный вектор терминов $v(d_i)$.
        *   Сохранить $v(d_i)$ в **инвертированном индексе**. Инвертированный индекс хранит для каждого термина список документов, в которых он встречается, и его вес (скор) в этих документах.
2.  **Поиск по запросу:**
    *   При поступлении запроса $Q$:
        *   Пропустить $Q$ через модель TILDE, чтобы получить его разреженный вектор терминов $v(Q)$.
        *   Используя $v(Q)$, выполнить поиск в инвертированном индексе.
        *   Вычислить скор релевантности для каждого кандидата $d_i$ как скалярное произведение $v(Q) \cdot v(d_i)$.
        *   Вернуть $K$ документов с наибольшими скорами.

Благодаря разреженному представлению, инференс TILDE по скорости сравним с BM25, но при этом обладает семантической мощью, унаследованной от Cross-Encoder.

### Результаты

На стандартных бенчмарках для ранжирования документов, таких как **MS MARCO Passage Ranking Dataset**, TILDE демонстрирует существенное улучшение по сравнению с традиционными разреженными методами:
*   TILDE превосходит BM25 на **20-30% по метрике MRR@10** (Mean Reciprocal Rank at 10), что является значительным скачком в качестве.
*   По сравнению с плотными методами (DPR, ANCE), TILDE показывает **сопоставимые, а иногда и превосходящие результаты по MRR@10**, при этом сохраняя преимущества разреженной индексации (меньший размер индекса, более быстрое обновление, лучшие характеристики при масштабировании).
*   TILDE демонстрирует **лучшую обобщающую способность (generalization)** на out-of-domain датасетах, где он может превосходить специально обученные плотные ретриверы, что указывает на то, что семантически обогащенные разреженные веса более устойчивы к изменениям в домене.
*   Важным результатом является то, что TILDE позволяет **интерпретировать предсказанные веса терминов**, что затруднено в чисто плотных моделях. Например, для запроса "как похудеть", TILDE может присвоить высокие веса терминам "диета" или "упражнения", даже если они не были в запросе, но семантически связаны.

## 📝 Критический анализ

```markdown
# TILDE (2021)
---
[[paper]](https://arxiv.org/pdf/2104.07223)<br>TILDE = Term Independent Learned DEcoders

TILDE — это подход к **нейронному разреженному поиску**, разработанный в Carnegie Mellon University. Он использует предварительно обученные языковые модели для создания семантически обогащенных разреженных представлений документов и запросов, сочетая эффективность разреженных методов с семантической глубиной плотных моделей.

### Контекст

Существуют два основных подхода в информационном поиске:
1. **Sparse Retrieval** (например, BM25, TF-IDF) — эффективны, но не учитывают семантические несоответствия.
2. **Dense Retrieval** (например, DPR, ANCE) — обеспечивают семантическое сопоставление, но требуют больших вычислительных ресурсов и сложны в обновлении.

Необходим метод, объединяющий семантическую мощь плотных моделей с эффективностью разреженных систем.

### Идея

TILDE использует Transformer-энкодер для **предсказания важности каждого термина** в документе или запросе, создавая **разреженные векторы терминов**. Это позволяет выполнять семантическое расширение, сохраняя разреженное представление для инвертированных индексов.

### Задача

Решается задача **ранжирования документов**: для запроса $Q$ и коллекции документов $D$, найти $K$ наиболее релевантных документов.

### Альтернативы

* **BM25 (1994):** Эффективен, но не чувствителен к семантике.
* **DPR (2020):** Улучшает семантическую релевантность, но имеет недостатки плотных методов.
* **ANCE (2020):** Улучшенная версия DPR.
* **ColBERT (2020):** Генерирует векторы для каждого токена.

TILDE генерирует **взвешенные разреженные векторы** из семантически богатой модели.

### Архитектура

1. **Transformer-энкодер:** Генерирует эмбеддинги для каждого токена.
2. **Проекционный слой:** Преобразует эмбеддинги в вектор размером словаря, представляющий **вес** каждого термина. Используется **максимум (max-pooling)** для агрегации.

<img src="img/img.png" width=500>

### Обучение

Используется **дистилляция знаний** от Cross-Encoder модели. TILDE минимизирует **KL-дивергенцию** между распределениями релевантности, предсказанными учителем и TILDE.

### Инференс

1. **Индексирование:** Генерация разреженных векторов для документов и сохранение в инвертированном индексе.
2. **Поиск:** Генерация разреженного вектора для запроса и поиск в индексе. Вычисление релевантности через скалярное произведение.

### Результаты

На MS MARCO Passage Ranking Dataset TILDE:
* Превосходит BM25 на **20-30% по MRR@10**.
* Сопоставим с плотными методами по MRR@10, сохраняя преимущества разреженной индексации.
* Демонстрирует **лучшую обобщающую способность** на out-of-domain датасетах.
* Позволяет **интерпретировать веса терминов**, что затруднено в плотных моделях.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# Устанавливаем устройство для вычислений
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Загружаем предварительно обученную модель и токенизатор BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').to(device)

# Пример документа и запроса
document = "The quick brown fox jumps over the lazy dog"
query = "fast fox"

# Токенизация документа и запроса
doc_tokens = tokenizer(document, return_tensors='pt').to(device)
query_tokens = tokenizer(query, return_tensors='pt').to(device)

# Получаем эмбеддинги из BERT
doc_embeddings = model(**doc_tokens).last_hidden_state
query_embeddings = model(**query_tokens).last_hidden_state

# Проекционный слой для генерации разреженных векторов
class SparseProjectionLayer(nn.Module):
    def __init__(self, vocab_size):
        super(SparseProjectionLayer, self).__init__()
        self.linear = nn.Linear(doc_embeddings.size(-1), vocab_size)

    def forward(self, embeddings):
        # Применяем линейный слой
        scores = self.linear(embeddings)
        # Применяем ReLU для разреженности
        sparse_vector = torch.relu(scores)
        return sparse_vector

# Инициализируем проекционный слой
vocab_size = len(tokenizer)  # Размер словаря
projection_layer = SparseProjectionLayer(vocab_size).to(device)

# Генерируем разреженные векторы для документа и запроса
doc_sparse_vector = projection_layer(doc_embeddings)
query_sparse_vector = projection_layer(query_embeddings)

# Агрегация с использованием max-pooling
doc_sparse_vector = torch.max(doc_sparse_vector, dim=1).values
query_sparse_vector = torch.max(query_sparse_vector, dim=1).values

# Вычисляем скалярное произведение для оценки релевантности
relevance_score = torch.dot(query_sparse_vector.squeeze(), doc_sparse_vector.squeeze())

print(f"Relevance Score: {relevance_score.item()}")

# Пример применения порога для разреженности
threshold = 0.1
doc_sparse_vector[doc_sparse_vector < threshold] = 0
query_sparse_vector[query_sparse_vector < threshold] = 0

# Пересчитываем скалярное произведение после применения порога
relevance_score_sparse = torch.dot(query_sparse_vector.squeeze(), doc_sparse_vector.squeeze())

print(f"Sparse Relevance Score: {relevance_score_sparse.item()}")
```

### Объяснение кода:

1. **Загрузка модели и токенизатора BERT:** Используем `bert-base-uncased` для получения эмбеддингов токенов.

2. **Токенизация и получение эмбеддингов:** Пропускаем документ и запрос через BERT, чтобы получить контекстуализированные эмбеддинги.

3. **Проекционный слой:** Создаем линейный слой, который проецирует эмбеддинги токенов в вектор размером с размер словаря. Это ключевая часть TILDE, где каждый токен получает вес для каждого термина в словаре.

4. **Агрегация:** Используем max-pooling для получения окончательного разреженного представления документа и запроса.

5. **Вычисление релевантности:** Скалярное произведение разреженных векторов документа и запроса дает оценку их релевантности.

6. **Пороговая разреженность:** Применяем порог для обнуления малых значений, что делает вектор еще более разреженным.

Этот код иллюстрирует, как TILDE использует мощь трансформеров для генерации семантически обогащенных разреженных представлений, сохраняя при этом эффективность традиционных методов разреженного поиска.